# E1 · Variance comparison

## 1. Purpose and design

E1 compares LF, OGM, SI, ICS, SOCP and VGMC on H₂, LiH, BeH₂, H₂O and NH₃. Each molecule uses the same Hamiltonian and reference state. Designs use either exact or CISD covariance. Their coefficients and shot fractions are evaluated with reference-state covariance. The two tables contain variance coefficients in Ha².

The common overlapping SI library defines the starting point for SOCP and VGMC. VGMC grows this library and reoptimizes its coefficients and allocations. ICS performs five updates through the referenced Tequila implementation. OGM uses its pooled estimator and integer measurement schedule.

The molecular Hamiltonians use the authors' fermionic files and the Bravyi–Kitaev transform. [Hamiltonian sources](../docs/sources.md#molecular-hamiltonians)

## 2. Imports and settings

`MODE = "analysis_only"` processes the supplied records. `MODE = "full"` executes the configured tasks in `RUN` and analyzes their commits. `MODE = "render"` loads an existing analysis. `RUN_MODE` selects fresh execution, compatible resume, verified reuse or automatic selection.


In [ ]:
from pathlib import Path
import logging

from IPython.display import display
from vgmc import plots
from vgmc.config import E1Config
from vgmc.data import read_json
from vgmc.workflow import analyze, load_analysis, render, run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "DATA_MANIFEST.json").is_file())
MODE = "analysis_only"
RUN_MODE = "fresh"
MOLECULES = ['H2', 'LiH', 'BeH2', 'H2O', 'NH3']
RUN = ROOT / "outputs" / "E1" / "run-001"
OUTPUT = ROOT / "outputs" / "E1" / "report"
ANALYSIS = OUTPUT / "analysis"
CONFIG = E1Config.from_mapping(
    read_json(ROOT / "configs" / "E1.json"))
logging.basicConfig(level=logging.INFO, format="%(message)s")
COVARIANCES = ["exact", "cisd"]
METHODS = list(CONFIG.methods)
WORKERS = 1


## 3. Run, collect and display

The analysis stores the selected rows and source hashes. Rendering reads that saved analysis.


In [ ]:
if MODE == "full":
    run_experiment("E1", ROOT, RUN, molecules=MOLECULES,
                   covariances=COVARIANCES, methods=METHODS, workers=WORKERS,
                   config=CONFIG, mode=RUN_MODE, allow_compute=True)
    analyze("E1", ROOT, ANALYSIS, run=RUN, molecules=MOLECULES)
elif MODE == "analysis_only":
    analyze("E1", ROOT, ANALYSIS, molecules=MOLECULES)
elif MODE != "render":
    raise ValueError("MODE must be analysis_only, full or render")

results = load_analysis("E1", ANALYSIS)
artifacts = render("E1", ANALYSIS, OUTPUT)


In [ ]:
for covariance in COVARIANCES:
    print(f"{covariance.upper()} design covariance · variance coefficient (Ha²)")
    display(plots.table_e1(results, covariance).style.format("{:.4g}"))
